# cAPTure development OOF early-warning audit

This notebook reuses the completed operational OOF report and its selected score thresholds. It does not train models, read packet Parquet files, select new thresholds, or access final-test scenarios.

An attack-step iteration is timely only if the first correct window-close alert occurs strictly before its last malicious packet. For each development scenario, the first correct chain alert is early only if it occurs strictly before the first packet of a declared terminal action. Terminal actions are frozen in `configs/capture_early_warning_audit_v1.yaml`. Each scenario is one observed chain episode, so chain-level rates are descriptive.


## 1. Prepare Colab


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from datetime import datetime, timezone
from pathlib import Path
import json
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")
if not PROJECT_ROOT.exists():
    subprocess.run(["git", "clone", "--branch", REPOSITORY_BRANCH,
                    "--single-branch", REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
branch = subprocess.check_output(["git", "branch", "--show-current"],
                                 cwd=PROJECT_ROOT, text=True).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f"Expected branch {REPOSITORY_BRANCH}, found {branch}.")
required_files = [
    "code/python/requirements-capture-xgb.txt",
    "code/python/utils/capture_early_warning.py",
    "code/python/utils/capture_oof_operational.py",
    "configs/capture_experiment_v1.yaml",
    "configs/capture_early_warning_audit_v1.yaml",
]
missing = [name for name in required_files if not (PROJECT_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f"Update the Colab repository copy first: {missing}")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(PROJECT_ROOT / "code/python/requirements-capture-xgb.txt")], check=True)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))
import pandas as pd
from IPython.display import display
print("Repository commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip())


## 2. Select the completed operational run

The default ID is the operational run recorded in the previous notebook. Change it if your completed run has another ID. Set `EARLY_WARNING_RUN_ID` only when resuming an existing audit.


In [ ]:
from utils.capture_early_warning import (
    run_early_warning_audit, validate_early_warning_audit,
)

MANIFEST_PATH = PROJECT_ROOT / "configs/capture_experiment_v1.yaml"
POLICY_PATH = PROJECT_ROOT / "configs/capture_early_warning_audit_v1.yaml"
OPERATIONAL_RUN_ID = "20260920T142514_611048Z_operational_oof"
EARLY_WARNING_RUN_ID = None  # Set only when resuming an existing audit.
if EARLY_WARNING_RUN_ID is None:
    EARLY_WARNING_RUN_ID = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
        + "_early_warning_oof"
    )
OPERATIONAL_DIR = DRIVE_ROOT / "operational_oof_runs" / OPERATIONAL_RUN_ID
OUTPUT_DIR = DRIVE_ROOT / "early_warning_oof_runs" / EARLY_WARNING_RUN_ID
print("Operational input:", OPERATIONAL_DIR)
print("Audit output:", OUTPUT_DIR)


## 3. Compute or verify the immutable audit


In [ ]:
audit_args = {
    "operational_dir": OPERATIONAL_DIR,
    "manifest_path": MANIFEST_PATH,
    "policy_path": POLICY_PATH,
    "output_dir": OUTPUT_DIR,
}
if OUTPUT_DIR.exists():
    report = validate_early_warning_audit(**audit_args)
else:
    report = run_early_warning_audit(**audit_args)
operational = json.loads(
    (OPERATIONAL_DIR / "operational_report.json").read_text(encoding="utf-8")
)
print("Early-warning audit run ID:", EARLY_WARNING_RUN_ID)
print("Terminal actions:", report["policy"]["terminal_action_steps"])


## 4. Compare warning times at every declared alert budget

The score-positive rate reproduces the original iteration detection rule. The timely rate requires the alert to arrive before the iteration's last malicious packet. The preterminal chain metric is a hierarchical macro of one episode-level flag per scenario; inspect the scenario table below rather than treating five episodes as a large sample.


In [ ]:
comparison_rows = []
for model_name, model_report in report["models"].items():
    for budget_name in report["budget_order"]:
        audit_budget = model_report["budgets"][budget_name]
        operational_budget = operational["models"][model_name]["budgets"][budget_name]
        comparison_rows.append({
            "model": model_name,
            "budget": budget_name,
            "threshold": model_report["thresholds"][budget_name]["threshold"],
            "false_alert_windows_per_hour": (
                operational_budget["hierarchical_macro"]["false_alert_windows_per_hour"]),
            **audit_budget["hierarchical_macro"],
        })
comparison = pd.DataFrame(comparison_rows).set_index(["model", "budget"])
display(comparison)


## 5. Inspect individual scenarios and terminal-action deadlines


In [ ]:
PRIMARY_BUDGET = "one_per_hour"
scenario_rows = []
chain_rows = []
for model_name, model_report in report["models"].items():
    audit_budget = model_report["budgets"][PRIMARY_BUDGET]
    for scenario, item in audit_budget["scenario_metrics"].items():
        scenario_rows.append({"model": model_name, **item})
    for scenario, item in audit_budget["chain_metrics"].items():
        chain_rows.append({"model": model_name, **item})
scenario_columns = [
    "fold", "iterations", "score_positive_iterations",
    "timely_iterations", "late_positive_iterations",
    "timely_iteration_rate", "early_before_terminal_action",
    "preterminal_opportunity_seconds",
]
display(pd.DataFrame(scenario_rows).set_index(["model", "scenario"])[scenario_columns])
chain_columns = [
    "first_terminal_action_step", "preterminal_opportunity_seconds",
    "first_correct_alert_steps", "early_before_terminal_action",
    "seconds_before_terminal_action",
    "seconds_at_or_after_terminal_action",
]
display(pd.DataFrame(chain_rows).set_index(["model", "scenario"])[chain_columns])


## 6. Inspect attack steps with late or missing alerts


In [ ]:
step_rows = []
for model_name, model_report in report["models"].items():
    for item in model_report["budgets"][PRIMARY_BUDGET]["step_metrics"].values():
        step_rows.append({"model": model_name, **item})
step_columns = [
    "model", "scenario", "attack_step", "iterations",
    "timely_iterations", "late_positive_iterations",
    "no_score_positive_iterations", "timely_iteration_rate",
]
step_table = pd.DataFrame(step_rows)[step_columns]
display(step_table.sort_values(
    ["timely_iteration_rate", "iterations"], ascending=[True, False]
).head(40))
print("Complete step and iteration details are stored in:",
      OUTPUT_DIR / "early_warning_report.json")


## Interpretation

A qualifying score after the last malicious packet is a late alert for that iteration. An alert after an earlier step can still be an early warning for the chain if it precedes the first declared terminal action. The chain table describes five development scenario episodes and does not estimate a population-level early-warning rate. The selected thresholds and OOF scores come from the same development data; final-test performance remains unmeasured.
